# 非線形バネをiLQRで制御する



## 制御モデル 非線形バネ

下記のバネ-マスモデルでバネ$F(x)$を非線形なモデルとして取り扱い、iLQRでどのように動かすのかを学習する。

<p align="center">
<img src="./images/nonlinear_spring.png" style="width:35%;" />
</p>


$x(t)$に対する運動方程式は次である。

$$
m \ddot{x}(t) = u(t) - k x(t) - k_3 x(t)^3
$$

$v(t) = \dot{x}(t)$とし、状態$X=[x(t) , v(t)]^T$とすると、非線形状態方式は以下となる。

$$
\begin{bmatrix}
\dot{x}(t) \\ \dot{v}(t)
\end{bmatrix} =
\begin{bmatrix}
v(t) \\
\frac{1}{m}\left( u(t) - k x(t) - k_3 x(t)^3 \right)
\end{bmatrix}
$$

前進オイラー法を用いて、上式は次の離散状態空間表現となる。$\Delta t$は離散時間である。

$$
\boxed{
\begin{bmatrix}
{x}_{n+1} \\ {v}_{n+1}
\end{bmatrix} =
\begin{bmatrix}
x_{n} + \Delta t v_{n} \\
v_{n} + \frac{\Delta t}{m}\left( u_n - k x_n - k_3 x_n^3 \right)
\end{bmatrix}
}
$$

上式は非線形であり$X_{n+1} = f(X_n , u_n)$として表す。




## コスト関数

総コスト$J$を次の構成とする。今回はステージコストは時刻によって変動させず、一定とする。

$$
J(X,U) = \phi(X_N) + \sum_{n=0}^{N-1} \ell(X_n, u_n)
$$

状態誤差を次のように設定する。

$$
e_x = x - x_{ref} , \quad e_v = v - v_{ref}
$$

終端では以下となる。

$$
e_{x,N} = x_N - x_{ref} , \quad e_{v,N} = v_N - v_{ref}
$$


入力目標値は、初期位置はゼロから目標状態で停止させるため、以下のようにする。

$$
u_{ref} = k \ x_{ref} + k_3 x_{ref}^3
$$ 

そして、入力誤差を次のように設定する。

$$
e_u = u - u_{ref}
$$

### ステージコスト $ell$

$$
\ell(X, u) = \frac{1}{2} q_x e_x^2 + \frac{1}{2} q_v e_v^2 + \frac{1}{2} r e_u^2 + \frac{1}{4} q_4 e_x^4
$$

$q_x, q_v, r, q_r$はそれぞれの誤差に対する重みであり、スカラーである。

各項の役割は次となる。

- $e_x^2$ : 位置誤差を小さくする
- $e_v^2$ : 速度誤差を小さくする
- $e_u^2$ : 入力誤差を小さくする
- $e_x^4$ : 位置誤差が大きく離れた状態を強く罰する

### 終端コスト $\phi$

$$
\phi(X_N) = \frac{1}{2} q_{x,N} e_{x,N}^2 + \frac{1}{2} q_{v,N} e_{v,N}^2 + \frac{1}{4} q_{4,N} e_{x,N}^4
$$

$q_{x,N}, q_{v,N}, q_{4,N}$は終端時刻での各誤差に対する重みであり、スカラーである。

- $e_{x,N}^2$ : 終端時刻の位置誤差を小さくする
- $e_{v,N}^2$ : 終端時刻の速度誤差を小さくする
- $e_{x,N}^4$ : 終端時刻の位置誤差が大きく離れた状態を強く罰する


## モデルの一次近似と、コスト関数の2次近似

#### 非線形モデルの一次近似

以下の非線形バネモデル$X_{n+1} = f(X_n , u_n)$を$X_n$と$u_n$で偏微分する。

$$
\begin{bmatrix}
\dot{x}_{n+1} \\ \dot{v}_{n+1}
\end{bmatrix} =
\begin{bmatrix}
x_{n+1} + \Delta t v_{n} \\
v_{n} + \frac{\Delta t}{m}\left( u_n - k x_n - k_3 x_n^3 \right)
\end{bmatrix}
$$

$$
\begin{aligned}
\frac{\partial f(X_n, u_n)}{\partial X_n} &= \begin{bmatrix}
\dfrac{\partial x_{n+1}}{\partial x_n} & \dfrac{\partial x_{n+1}}{\partial v_n} \\[8pt]
\dfrac{\partial v_{n+1}}{\partial x_n} & \dfrac{\partial v_{n+1}}{\partial v_n} \\[8pt]
\end{bmatrix} \\
&= \begin{bmatrix}
1  & \Delta t \\[8pt]
-\dfrac{\Delta t}{m}(k + 3 k_3 x_n^2) & 1
\end{bmatrix}
\end{aligned}
$$

$$
\begin{aligned}
\frac{\partial f(X_n, u_n)}{\partial u_n} &= \begin{bmatrix}
\dfrac{\partial x_{n+1}}{\partial u_n}  \\[8pt]
\dfrac{\partial v_{n+1}}{\partial u_n} 
\end{bmatrix} \\
&= \begin{bmatrix}
0 \\[8pt]
\dfrac{\Delta t}{m}
\end{bmatrix}
\end{aligned}
$$

ここから、一次近似摂動モデル$\delta X_{n+1}$を以下にように構成する。

$$
\delta X_{n+1} \approx A_n \delta X_n + B_n \delta u_n
$$

$$
\boxed{
A_n = \left. \frac{\partial f(X_n, u_n)}{\partial X_n} \right|_{(\bar{X}_n, \bar{u}_n)} = \begin{bmatrix}
1  & \Delta t \\[8pt]
-\dfrac{\Delta t}{m}(k + 3 k_3 \bar{x}_n^2) & 1
\end{bmatrix}}
$$

$$
\boxed{
B_n = \left. \frac{\partial f(X_n, u_n)}{\partial u_n} \right|_{\bar{X}_n, \bar{u}_n)} =\begin{bmatrix}
0 \\[8pt]
\dfrac{\Delta t}{m}
\end{bmatrix}}
$$


#### コスト関数の二次近似

ステージコスト $\ell$と終端コスト$\phi$をそれぞれ二次近似するが、必要になのは、Taylor展開の1次項と2次項である。

$$
\ell(X, u) = \frac{1}{2} q_x e_x^2 + \frac{1}{2} q_v e_v^2 + \frac{1}{2} r e_u^2 + \frac{1}{4} q_4 e_x^4
$$

$$
\phi(X_N) = \frac{1}{2} q_{x,N} e_{x,N}^2 + \frac{1}{2} q_{v,N} e_{v,N}^2 + \frac{1}{4} q_{4,N} e_{x,N}^4
$$

$$
e_x = x - x_{ref} , \quad e_v = v - v_{ref}, \quad e_u = u - u_{ref}
$$

$$
e_{x,N} = x_N - x_{ref} , \quad e_{v,N} = v_N - v_{ref}
$$

##### ステージコストの２次近似係数

$$
\begin{aligned}
\ell_{X,n} &=
\left. \frac{\partial \ell(X,u)}{\partial X} \right|_{(\bar{X}_n, \bar{u}_n)}=
\left. \begin{bmatrix} 
\dfrac{\partial \ell(X,u)}{\partial x} \\[8pt]
\dfrac{\partial \ell(X,u)}{\partial v}
\end{bmatrix}  \right|_{(\bar{X}_n, \bar{u}_n)} =
\left. \begin{bmatrix}
q_x e_x + q_4 e_x^3 \\
q_v e_v
\end{bmatrix}  \right|_{(\bar{X}_n, \bar{u}_n)} =
\begin{bmatrix}
q_x (\bar{x}_n - x_{ref}) + q_4 (\bar{x}_n - x_{ref})^3 \\
q_v (\bar{v}_n - v_{ref})
\end{bmatrix} \\
\ell_{u,n} &= \left. \frac{\partial \ell(X,u)}{\partial u} \right|_{(\bar{X}_n, \bar{u}_n)} = \left. r e_u \right|_{(\bar{X}_n, \bar{u}_n)} = r (\bar{u} - u_{ref}) \\
\ell_{XX,n} &= \left. \frac{\partial^2 \ell(X,u)}{\partial X^2} \right|_{(\bar{X}_n, \bar{u}_n)}=
\left. \begin{bmatrix} 
\dfrac{\partial^2 \ell(X,u)}{\partial x^2} & \dfrac{\partial^2 \ell(X,u)}{\partial x \partial v} \\[8pt]
\dfrac{\partial^2 \ell(X,u)}{\partial v \partial x} & \dfrac{\partial^2 \ell(X,u)}{\partial v^2}
\end{bmatrix} \right|_{(\bar{X}_n, \bar{u}_n)} = 
\left. \begin{bmatrix}
q_x + 3 q_4 e_x^2 & 0 \\
0 & q_v
\end{bmatrix} \right|_{(\bar{X}_n, \bar{u}_n)} =
\begin{bmatrix}
q_x + 3 q_4 (\bar{x}_n - x_{ref})^2 & 0 \\
0 & q_v
\end{bmatrix} \\
\ell_{Xu,n} &= \left. \frac{\partial}{\partial u} \frac{\partial \ell(X,u)}{\partial X} \right|_{(\bar{X}_n, \bar{u}_n)}= 
\left. \begin{bmatrix} 
\dfrac{\partial^2 \ell(X,u)}{\partial u \partial x} & \dfrac{\partial^2 \ell(X,u)}{\partial u\partial v}
\end{bmatrix}  \right|_{(\bar{X}_n, \bar{u}_n)} = \begin{bmatrix} 0 & 0 \end{bmatrix} \\
\ell_{uu,n} &= \left. \frac{\partial^2 \ell(X,u)}{\partial u^2} \right|_{(\bar{X}_n, \bar{u}_n)} = r
\end{aligned}
$$




#### 終端コストの2次近似


$$
\begin{aligned}
\phi_X &= \left. \frac{\partial \phi}{\partial X_N} \right|_{\bar{X}_N} = 
\left. \begin{bmatrix}
\dfrac{\partial \phi}{\partial x_N} \\ \dfrac{\partial \phi}{\partial v_N}
\end{bmatrix}  \right|_{\bar{X}_N} =
\left. \begin{bmatrix}
q_{x,N} e_{x,N} + q_{4,N} e_{x,N}^3 \\ q_{v,N} e_{v,N}
\end{bmatrix}  \right|_{\bar{X}_N} =
\begin{bmatrix}
q_{x,N} (\bar{x}_N - x_{ref}) + q_{4,N} (\bar{x}_N - x_{ref})^3 \\ q_{v,N} (\bar{v}_N - v_{ref})
\end{bmatrix} \\
\phi_{XX} &= \left. \frac{\partial^2 \phi}{\partial X_N^2} \right|_{\bar{X}_N} = 
\left. \begin{bmatrix}
\dfrac{\partial^2 \phi}{\partial x_N^2} & \dfrac{\partial^2 \phi}{\partial v_N \partial x_N }\\
\dfrac{\partial^2 \phi}{\partial x_N \partial v_N} & \dfrac{\partial^2 \phi}{\partial v_N^2}
\end{bmatrix}  \right|_{\bar{X}_N} =
\left. \begin{bmatrix}
q_{x,N} + 3 q_{4,N} e_{x,N}^2 & 0 \\
0 & q_{v,N}
\end{bmatrix}  \right|_{\bar{X}_N} =
\begin{bmatrix}
q_{x,N} + 3 q_{4,N} (\bar{x}_N - x_{ref})^2 & 0 \\
0 & q_{v,N}
\end{bmatrix}
\end{aligned}
$$

